# Step 3: logistic regression

Features: the last 5 daily returns and yesterday's direction. Each market runs through the six walk-forward folds (test years 2017–2022). Part A uses sklearn; Part B is the same model in PyTorch, checked against sklearn.

In [1]:
import numpy as np
import pandas as pd

from data import load_all
from models import fit_sklearn, fit_torch, pooled, report, walk_forward

markets = load_all()
C_GRID = [0.0001, 0.001, 0.01, 0.1, 1, 10]   # sklearn: smaller C = stronger penalty
WD_GRID = [0.0001, 0.001, 0.01, 0.1, 1, 10]  # PyTorch: larger weight decay = stronger penalty

## Part A: sklearn

Per test year for India (change the name to see another market):

In [2]:
sk = {market: walk_forward(df, fit_sklearn, C_GRID) for market, df in markets.items()}
report(sk["India"]).round(3)

,setting,accuracy %,MCC,AUC,log loss,fair bar %,p vs fair,stretch bar %,stretch guess,p vs stretch
year,,,,,,,,,,
2017,0.000,56.275,0.000,0.519,0.689,56.275,1.000,56.275,always up,1.000
2018,10.000,56.504,0.102,0.555,0.687,54.065,0.497,54.065,always up,0.497
2019,0.100,53.469,0.040,0.562,0.689,53.061,1.000,53.878,same as yesterday,1.000
2020,0.010,53.571,-0.029,0.526,0.693,59.127,0.092,59.127,always up,0.092
2021,0.000,55.645,0.000,0.539,0.687,55.645,1.000,56.855,same as yesterday,0.850
2022,0.001,48.387,-0.082,0.569,0.692,50.000,0.344,59.274,same as yesterday,0.014


All six test years pooled, per market:

In [3]:
pd.DataFrame({market: pooled(p) for market, p in sk.items()}).T.round(3)

,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday
Indonesia,52.881,0.010,0.514,52.881,1.000,49.520,0.067
Malaysia,49.523,-0.013,0.481,49.795,0.749,49.250,0.909
Philippines,51.540,0.007,0.492,51.540,1.000,49.008,0.177
Singapore,53.711,0.070,0.534,51.286,0.110,49.229,0.018
Thailand,52.093,-0.056,0.495,52.437,0.062,50.995,0.567
India,53.970,0.010,0.544,54.711,0.442,54.980,0.535


## Part B: PyTorch

Takes about 2 minutes.

In [4]:
tc = {market: walk_forward(df, fit_torch, WD_GRID) for market, df in markets.items()}
pd.DataFrame({market: pooled(p) for market, p in tc.items()}).T.round(3)

,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday
Indonesia,52.812,0.004,0.516,52.881,1.000,49.520,0.073
Malaysia,49.932,0.005,0.480,49.795,0.911,49.250,0.733
Philippines,51.472,0.003,0.490,51.540,1.000,49.008,0.190
Singapore,53.123,0.057,0.533,51.286,0.213,49.229,0.041
Thailand,52.093,-0.056,0.494,52.437,0.062,50.995,0.567
India,54.643,0.034,0.547,54.711,1.000,54.980,0.856


## Do the two versions agree?

Target: correlation above 0.99 in every fold, and almost all up/down calls the same.

In [5]:
rows = []
for market in markets:
    both = sk[market][["year", "p_up"]].join(tc[market]["p_up"], rsuffix="_torch")
    for year, f in both.groupby("year"):
        rows.append({
            "market": market,
            "year": year,
            "correlation": np.corrcoef(f["p_up"], f["p_up_torch"])[0, 1],
            "calls agree %": 100 * ((f["p_up"] > 0.5) == (f["p_up_torch"] > 0.5)).mean(),
        })
agreement = pd.DataFrame(rows)
agreement.round(4)

C:\Users\kesav\AppData\Local\Temp\ipykernel_29388\2785015729.py:3: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  both = sk[market][["year", "p_up"]].join(tc[market]["p_up"], rsuffix="_torch")


,market,year,correlation,calls agree %
0,Indonesia,2017,0.9997,99.1597
1,Indonesia,2018,0.9998,98.7500
2,Indonesia,2019,0.9995,99.1837
3,Indonesia,2020,0.9999,98.7603
4,Indonesia,2021,1.0000,100.0000
5,Indonesia,2022,0.9990,98.7805
6,Malaysia,2017,1.0000,99.5885
7,Malaysia,2018,1.0000,100.0000
8,Malaysia,2019,0.9997,97.1311
9,Malaysia,2020,1.0000,100.0000


In [6]:
print(f"lowest correlation:     {agreement['correlation'].min():.4f}")
print(f"lowest calls agreement: {agreement['calls agree %'].min():.1f}%")

lowest correlation:     0.9986
lowest calls agreement: 89.5%
